<a href="https://colab.research.google.com/github/kej534923-maker/card-1995-iv-replication/blob/main/02_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 11.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

In [17]:
import re

def extract_varlist_from_sas(path="read1.sas"):
    text = open(path, "r", errors="ignore").read()

    m = re.search(r"(?mi)^\s*input\s+(.*?);", text, flags=re.DOTALL)
    if not m:
        raise ValueError("Cannot find INPUT block (line-start input ... ;)")

    block = m.group(1)

    block = re.sub(r"/\*.*?\*/", " ", block, flags=re.DOTALL)

    tokens = re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", block)

    return tokens

colnames = extract_varlist_from_sas("read1.sas")
print("Number of variables:", len(colnames))
print("First 10:", colnames[:10])
print("Last 5:", colnames[-5:])

Number of variables: 52
First 10: ['id', 'nearc2', 'nearc4', 'nearc4a', 'nearc4b', 'ed76', 'ed66', 'age76', 'daded', 'nodaded']
Last 5: ['iq', 'marsta76', 'marsta78', 'marsta80', 'libcrd14']


In [18]:
import pandas as pd

df = pd.read_csv(
    "nls.dat",
    sep=r"\s+",
    header=None,
    names=colnames,
    engine="python"
)

df["age76_sq"] = df["age76"] ** 2

print(df.shape)
df.head()

(3613, 53)


,id,nearc2,nearc4,nearc4a,nearc4b,ed76,ed66,age76,daded,nodaded,...,enroll76,enroll78,enroll80,kww,iq,marsta76,marsta78,marsta80,libcrd14,age76_sq
0,2,0,0,0,0,7,5,29,9.94,1,...,0,0,0,15,.,1,1,1,0,841
1,3,0,0,0,0,12,11,27,8.00,0,...,0,0,0,35,93,1,4,4,1,729
2,4,0,0,0,0,12,12,34,14.00,0,...,0,.,.,42,103,1,.,.,1,1156
3,5,1,1,1,0,11,11,27,11.00,0,...,0,.,0,25,88,1,.,5,1,729
4,6,1,1,1,0,12,12,34,8.00,0,...,0,0,.,34,108,1,1,.,0,1156


In [21]:
import numpy as np
import pandas as pd

# 把 '.' 当作缺失值
df = df.replace(".", np.nan)

# 把所有列都强制转成 numeric（转不了的变 NaN）
df = df.apply(pd.to_numeric, errors="coerce")

In [22]:
import statsmodels.api as sm

cols = ["lwage76", "ed76", "black", "south66", "smsa66r", "age76", "age76_sq"]
df_reg = df[cols].dropna()

X = df_reg[["ed76", "black", "south66", "smsa66r", "age76", "age76_sq"]]
X = sm.add_constant(X)

y = df_reg["lwage76"]

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                lwage76   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.256
Method:                 Least Squares   F-statistic:                     174.0
Date:                Sat, 28 Feb 2026   Prob (F-statistic):          2.13e-190
Time:                        00:53:15   Log-Likelihood:                -1376.2
No. Observations:                3010   AIC:                             2766.
Df Residuals:                    3003   BIC:                             2808.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9490      0.654      4.506      0.0